# Reconstruction pipeline

This notebook demonstrates the modular reconstruction pipeline used to turn raw k-space
data into images.

The Cartesian pipeline composes zero filling, FFT, homodyne and POCS data consistency.
Variational reconstruction is covered separately in
`variational_reconstruction.ipynb`.

The examples below use small synthetic data.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from mrax.data_struct import AcquisitionData, ReconstructionRecipe
from mrax.reco import ReconstructionPipeline, fftnc, ifftnc

# Synthetic object and fully-sampled k-space
time_series = np.zeros((1, 32, 32), dtype=np.float32)
time_series[0, 12:20, 12:20] = 1.0
kspace_full = fftnc(time_series)

# Partial Fourier undersampling (PF=0.75): keep the last 3/4 of ky lines.
mask = np.zeros((32, 32), dtype=bool)
mask[8:, :] = True
undersampled = kspace_full * mask[None, ...]

# Baseline: zero-filled inverse FFT.
zf_img = np.abs(ifftnc(undersampled))[0]

acquisition = AcquisitionData(np.asarray(undersampled)[:, None], (32, 32), ("phase", "read"),
    acquired=mask, recipe=ReconstructionRecipe(partial_fourier="pocs", partial_fourier_iterations=20))
pipeline = ReconstructionPipeline.from_acquisition(acquisition)
result = pipeline.run(acquisition)
reco_img = np.asarray(result.image)[0]
print(f"Samples: {acquisition.samples.shape}; image: {result.image.shape}")

vmax = float(np.max(time_series))
fig, axes = plt.subplots(1, 3, figsize=(12, 3), constrained_layout=True)
axes[0].imshow(time_series[0], cmap="gray", vmin=0, vmax=vmax)
axes[0].set_title("Ground truth")
axes[1].imshow(zf_img, cmap="gray", vmin=0, vmax=vmax)
axes[1].set_title("Zero-filled")
axes[2].imshow(reco_img, cmap="gray", vmin=0, vmax=vmax)
axes[2].set_title("POCS")
for ax in axes:
    ax.axis("off")
plt.show()


### Fourier transform sanity check

`ifftnc` provides the centered orthonormal transform used by the acquisition pipeline.


In [ ]:
import jax.numpy as jnp

from mrax.reco import ifftnc

zf_np = np.fft.fftshift(np.fft.ifftn(np.fft.ifftshift(undersampled, axes=(-2, -1)), axes=(-2, -1), norm="ortho"), axes=(-2, -1))[0]
zf_jax = np.asarray(ifftnc(jnp.asarray(undersampled)))[0]

print(f"Max abs diff (NumPy vs JAX IFFT): {np.max(np.abs(zf_np - zf_jax)):.3e}")

fig, axes = plt.subplots(1, 2, figsize=(8, 3), constrained_layout=True)
axes[0].imshow(np.abs(zf_np), cmap="gray")
axes[0].set_title("Zero-filled (NumPy)")
axes[1].imshow(np.abs(zf_jax), cmap="gray")
axes[1].set_title("Zero-filled (JAX)")
for ax in axes:
    ax.axis("off")
plt.show()
